# Import dependencies

In [ ]:
%%capture
!pip install datasets evaluate transformers
!pip install https://gitlab.com/trungtv/vi_spacy/-/raw/master/packages/vi_core_news_lg-3.6.0/dist/vi_core_news_lg-3.6.0.tar.gz
!pip install pyvi rouge_score bert_score
!pip install --upgrade nltk

In [ ]:
import re
import gc
import nltk
import spacy
import torch
import wandb
import evaluate
import datasets
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datasets import Dataset, load_metric, load_dataset
from huggingface_hub import login
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

nltk.download("wordnet")
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('omw-1.4')
rouge = load_metric('rouge')
meteor = evaluate.load('meteor')
bertscore = evaluate.load("bertscore")
nlp = spacy.load('vi_core_news_lg')
login('your_huggingface_auth_token_here')
wandb.login(key='your_wandb_auth_token_here')

In [ ]:
%%capture
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

# Prepare data

In [ ]:
df = pd.read_csv(r'./vilegallm-syllogism-legal-reasoning/ViLegalQwen2.5-1.5B-Base.csv')
df

In [ ]:
def remove_reasoning_content(raw_text):
    if "</think>\n\n" in raw_text:
        return raw_text.split("</think>\n\n")[1]
    else:
        return ""

df["predictions"] = df["generated_answer"].apply(remove_reasoning_content)
df = df[["answer", "predictions"]]
df

In [ ]:
df.columns = ["references", "predictions"]
df

In [ ]:
df.dropna(subset=['predictions'], inplace=True)
df

In [ ]:
df['predictions'][5]

In [ ]:
df.to_csv(r'./working/results.csv', index=False, encoding='utf-8-sig')

# Model Evaluation

In [ ]:
def load_data(predictions_file, references_file):
    with open(predictions_file, 'r') as file:
        predictions = pd.read_csv(file)['predictions'].astype('str')
    with open(references_file, 'r') as file:
        references = pd.read_csv(file)['references'].astype('str')
    return predictions, references

def calculate_bleu(predictions, references):
    nlp = spacy.load('vi_core_news_lg')
    bleu_scores = {1: [], 2: [], 3: [], 4: []}
    weights = [(1, 0, 0, 0), (0.5, 0.5, 0, 0), (0.33, 0.33, 0.33, 0), (0.25, 0.25, 0.25, 0.25)]
    
    for pred, ref in zip(predictions, references):
        pred_tokens = [token.text for token in nlp(pred)]
        ref_tokens = [token.text for token in nlp(ref)]
        smoothing = SmoothingFunction().method1
        
        for n, weight in enumerate(weights, start=1):
            score = sentence_bleu([ref_tokens], pred_tokens, weights=weight, smoothing_function=smoothing)
            bleu_scores[n].append(score)
    
    return {f"BLEU-{n}": (sum(scores) / len(scores)) * 100 for n, scores in bleu_scores.items()}

def calculate_metrics(predictions, references):
    rouge = load_metric('rouge')
    meteor = evaluate.load('meteor')
    bertscore = evaluate.load("bertscore")
    
    rouge_scores = rouge.compute(predictions=predictions, references=references)
    rouge_results = {k: v.mid.fmeasure * 100 for k, v in rouge_scores.items()}
    
    meteor_score = meteor.compute(predictions=predictions, references=references)['meteor'] * 100
    
    bertscore_results = bertscore.compute(predictions=predictions, references=references, lang='vi')
    bertscore_f1 = np.mean(bertscore_results['f1']) * 100
    
    return {**rouge_results, 'METEOR': meteor_score, 'BERTScore-F1': bertscore_f1}

def evaluate_qa(predictions, references):
    bleu_scores = calculate_bleu(predictions, references)
    other_metrics = calculate_metrics(predictions, references)
    
    return {**bleu_scores, **other_metrics}

def main():
    predictions, references = load_data('./working/results.csv', './working/results.csv')
    results = evaluate_qa(predictions, references)
    
    print("\nQuestion Answering Evaluation Results:")
    for metric, value in results.items():
        print(f"{metric}: {value:.2f}")

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
main()